# Patient embeddings and supervised symptom training

This notebook is designed to run top to bottom in Google Colab. Select **Runtime → Change runtime type → GPU** before starting.

`LEGACY_REPRODUCTION_MODE=True` is the default and runs Workflow A: the isolated, faithful legacy procedure. Set it to `False` to run Workflow B, the existing modern learning-curve experiment. The two workflows use separate embedding, checkpoint, and result directories and never overwrite one another.

**Cells marked EDIT ME require configuration.**

## 1. Check the Colab GPU

Embedding generation later requires CUDA. The supervised model automatically uses CUDA when available and otherwise falls back to CPU.

In [ ]:
import platform
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "CPU"
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {CUDA_AVAILABLE}")
print(f"Detected device: {DEVICE_NAME}")
if not CUDA_AVAILABLE:
    print("WARNING: Enable a GPU runtime before running the embedding section.")

## 2. Clone or update the project repository — EDIT ME

Replace `YOUR_USERNAME/YOUR_REPOSITORY` with the GitHub repository containing `prepare_patient_embeddings.py` and `requirements.txt`. For a private repository, configure Colab GitHub credentials before running this cell.

In [ ]:
from pathlib import Path
import subprocess

# EDIT ME: replace this placeholder with the real repository URL.
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git"
PROJECT_DIR = Path("/content/Text2DAG")

if "YOUR_USERNAME/YOUR_REPOSITORY" in REPO_URL:
    raise ValueError("Edit REPO_URL before running this cell.")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True
    )
elif PROJECT_DIR.exists():
    raise RuntimeError(f"{PROJECT_DIR} exists but is not a Git repository.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project ready at {PROJECT_DIR}")

## 3. Install dependencies

This installs the repository requirements plus scikit-learn, iterative multi-label stratification, and matplotlib for splitting, evaluation, and the final learning-curve plot.

In [ ]:
import sys

requirements_path = PROJECT_DIR / "requirements.txt"
if not requirements_path.is_file():
    raise FileNotFoundError(f"Missing requirements file: {requirements_path}")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(requirements_path),
        "scikit-learn>=1.4,<2",
        "iterative-stratification>=0.1.9,<1",
        "matplotlib>=3.8,<4",
    ],
    check=True,
)
print("Dependencies installed.")


## 4. Mount Google Drive

Approve the Google Drive authorization prompt. All generated artifacts are stored in Drive so they survive Colab runtime resets.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 5. Configure Google Drive paths — EDIT ME

Edit `DRIVE_ROOT`, `MAPPING_PATH`, and `SOURCE_PATH` to match your Drive layout. The sentence-embedding, checkpoint, and evaluation directories are configurable independently. Embeddings are saved to Google Drive and reused automatically. Set `FORCE_RECOMPUTE=True` only when the source text, mapping, or embedding model changes.

In [ ]:
# EDIT ME: configure these Google Drive paths.
DRIVE_ROOT = Path("/content/drive/MyDrive/Text2DAG")
MAPPING_PATH = DRIVE_ROOT / "inputs/gfs_sentence_mapping.csv"
SOURCE_PATH = DRIVE_ROOT / "inputs/SynSUM.csv"

# Workflow A is the default because reproduction takes priority over correction.
LEGACY_REPRODUCTION_MODE = True
FORCE_RECOMPUTE = False
EMBEDDING_BATCH_SIZE = 32
EMBEDDING_MODEL_NAME = "nomic-ai/modernbert-embed-base"
RANDOM_SEED = 42  # Workflow B only; Workflow A fixes the legacy seed at 5.

# Workflow A: exact legacy reproduction. These paths never overlap modern outputs.
LEGACY_ROOT = DRIVE_ROOT / "legacy_reproduction"
LEGACY_EMBEDDING_OUTPUT_DIR = LEGACY_ROOT / "embeddings"
LEGACY_MODEL_CHECKPOINT_DIR = LEGACY_ROOT / "checkpoints"
LEGACY_EVALUATION_RESULTS_DIR = LEGACY_ROOT / "results"

# Workflow B: current learning-curve protocol.
EMBEDDING_OUTPUT_DIR = DRIVE_ROOT / "generated_sentence_embeddings"
MODEL_CHECKPOINT_DIR = DRIVE_ROOT / "model_checkpoints"
EVALUATION_RESULTS_DIR = DRIVE_ROOT / "evaluation_results"

selected_directories = (
    [LEGACY_EMBEDDING_OUTPUT_DIR, LEGACY_MODEL_CHECKPOINT_DIR, LEGACY_EVALUATION_RESULTS_DIR]
    if LEGACY_REPRODUCTION_MODE
    else [EMBEDDING_OUTPUT_DIR, MODEL_CHECKPOINT_DIR, EVALUATION_RESULTS_DIR]
)
for directory in selected_directories:
    directory.mkdir(parents=True, exist_ok=True)
required_inputs = [SOURCE_PATH] if LEGACY_REPRODUCTION_MODE else [MAPPING_PATH, SOURCE_PATH]
for input_path in required_inputs:
    if not input_path.is_file():
        raise FileNotFoundError(
            f"Input not found: {input_path}. Edit the path configuration cell."
        )

EMBEDDING_NPZ_PATH = EMBEDDING_OUTPUT_DIR / "patient_embeddings_and_labels.npz"
print(f"LEGACY_REPRODUCTION_MODE={LEGACY_REPRODUCTION_MODE}")
print(f"SynSUM: {SOURCE_PATH}")
if LEGACY_REPRODUCTION_MODE:
    print(f"Legacy embeddings: {LEGACY_EMBEDDING_OUTPUT_DIR}")
    print(f"Legacy checkpoints: {LEGACY_MODEL_CHECKPOINT_DIR}")
    print(f"Legacy results: {LEGACY_EVALUATION_RESULTS_DIR}")
else:
    print(f"Mapping: {MAPPING_PATH}")
    print(f"Modern embeddings: {EMBEDDING_OUTPUT_DIR}")
    print(f"Modern checkpoints: {MODEL_CHECKPOINT_DIR}")
    print(f"Modern evaluation results: {EVALUATION_RESULTS_DIR}")

# Workflow dispatch — legacy reproduction or modern embeddings

With `LEGACY_REPRODUCTION_MODE=True`, this cell runs the complete isolated Workflow A from raw `SynSUM.csv` through embeddings, training, predictions, metrics, split IDs, and metadata. It intentionally preserves raw fever value `2`, batch-local zero-sentence padding, unmasked sentence averaging, shuffled training-cache construction, and the two-layer linear head.

With the mode disabled, the original modern SentenceTransformer embedding workflow runs unchanged. Its artifacts remain in the modern directories.

In [ ]:
if LEGACY_REPRODUCTION_MODE:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required for faithful legacy ModernBERT embedding generation. "
            "Select Runtime > Change runtime type > GPU and rerun."
        )
    legacy_script = PROJECT_DIR / "legacy_reproduction.py"
    if not legacy_script.is_file():
        raise FileNotFoundError(f"Missing legacy reproduction script: {legacy_script}")
    command = [
        sys.executable,
        str(legacy_script),
        "--source-path", str(SOURCE_PATH),
        "--embedding-dir", str(LEGACY_EMBEDDING_OUTPUT_DIR),
        "--checkpoint-dir", str(LEGACY_MODEL_CHECKPOINT_DIR),
        "--results-dir", str(LEGACY_EVALUATION_RESULTS_DIR),
        "--model-name", EMBEDDING_MODEL_NAME,
        "--device", "cuda",
    ]
    if FORCE_RECOMPUTE:
        command.append("--force-recompute")
    print("Running Workflow A:", " ".join(command))
    subprocess.run(command, check=True, cwd=PROJECT_DIR)
    LEGACY_METADATA_PATH = (
        LEGACY_EVALUATION_RESULTS_DIR / "legacy_reproduction_metadata.json"
    )
    if not LEGACY_METADATA_PATH.is_file():
        raise FileNotFoundError(
            f"Expected legacy metadata was not created: {LEGACY_METADATA_PATH}"
        )
    print(f"Workflow A complete: {LEGACY_METADATA_PATH}")
else:
    if EMBEDDING_NPZ_PATH.exists() and not FORCE_RECOMPUTE:
        print(f"Skipping embedding generation; output exists: {EMBEDDING_NPZ_PATH}")
    else:
        if not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA is required for this notebook's embedding step. "
                "Select Runtime > Change runtime type > GPU and rerun."
            )
        preprocessing_script = PROJECT_DIR / "prepare_patient_embeddings.py"
        if not preprocessing_script.is_file():
            raise FileNotFoundError(f"Missing preprocessing script: {preprocessing_script}")

        command = [
            sys.executable,
            str(preprocessing_script),
            "--mapping-path", str(MAPPING_PATH),
            "--source-path", str(SOURCE_PATH),
            "--output-dir", str(EMBEDDING_OUTPUT_DIR),
            "--model-name", EMBEDDING_MODEL_NAME,
            "--batch-size", str(EMBEDDING_BATCH_SIZE),
            "--device", "cuda",
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, check=True, cwd=PROJECT_DIR)

    if not EMBEDDING_NPZ_PATH.is_file():
        raise FileNotFoundError(f"Expected embedding output was not created: {EMBEDDING_NPZ_PATH}")
    print(f"Embedding arrays ready: {EMBEDDING_NPZ_PATH}")

## Load, validate, and binarize the saved arrays

Start from this cell when rerunning only supervised training in a later Colab session (after mounting Drive and configuring the paths).

The source data may encode fever as `0=none`, `1=low`, and `2=high`. For the supervised experiment below, fever is explicitly converted to binary: `0=no fever`, `1=any fever` (both original 1 and 2 become 1). The original embedding file is not modified.


In [ ]:
if LEGACY_REPRODUCTION_MODE:
    print(
        "Modern array loading and fever binarization skipped. Workflow A used raw "
        "legacy targets and wrote its own diagnostics and artifacts."
    )
else:
    import numpy as np

    with np.load(EMBEDDING_NPZ_PATH, allow_pickle=False) as saved:
        required_keys = {"patient_ids", "X", "y", "label_names"}
        missing_keys = required_keys - set(saved.files)
        if missing_keys:
            raise ValueError(f"Embedding NPZ is missing keys: {sorted(missing_keys)}")
        patient_ids = saved["patient_ids"].astype(np.int64, copy=False)
        X = saved["X"].astype(np.float32, copy=False)
        y_original = saved["y"].astype(np.int64, copy=False)
        label_names = saved["label_names"].astype(str)

    EXPECTED_LABEL_NAMES = np.array(["dysp", "cough", "pain", "fever", "nasal"])
    if X.ndim != 2 or X.shape[1] != 768:
        raise ValueError(f"Expected X shape (n_patients, 768), found {X.shape}.")
    if y_original.ndim != 2 or y_original.shape[1] != 5:
        raise ValueError(f"Expected y shape (n_patients, 5), found {y_original.shape}.")
    if X.shape[0] != y_original.shape[0] or X.shape[0] != len(patient_ids):
        raise ValueError(
            f"Row mismatch: X={X.shape}, y={y_original.shape}, patient_ids={patient_ids.shape}."
        )
    if not np.array_equal(label_names, EXPECTED_LABEL_NAMES):
        raise ValueError(f"Unexpected label order: {label_names.tolist()}")
    if len(np.unique(patient_ids)) != len(patient_ids):
        raise ValueError("Duplicate patient IDs found in saved arrays.")
    if not np.isfinite(X).all():
        raise ValueError("X contains NaN or infinite values.")
    if not np.isfinite(y_original).all():
        raise ValueError("y contains NaN or infinite values.")
    for index in [0, 1, 2, 4]:
        if not set(np.unique(y_original[:, index])).issubset({0, 1}):
            raise ValueError(f"Binary label {label_names[index]} contains invalid values.")
    if not set(np.unique(y_original[:, 3])).issubset({0, 1, 2}):
        raise ValueError("Fever contains values outside {0, 1, 2}.")

    y = y_original.copy()
    y[:, 3] = (y_original[:, 3] > 0).astype(np.int64)
    if not set(np.unique(y)).issubset({0, 1}):
        raise ValueError("All supervised targets must be binary after fever conversion.")

    print(f"patient_ids: {patient_ids.shape}, {patient_ids.dtype}")
    print(f"X: {X.shape}, {X.dtype}")
    print(f"y: {y.shape}, {y.dtype}")
    print(f"label_names: {label_names.tolist()}")
    print("Original fever counts (none/low/high):", np.bincount(y_original[:, 3], minlength=3))
    print("Binary fever counts (no/yes):", np.bincount(y[:, 3], minlength=2))


# Part B — Binary multi-task symptom learning curve

This section treats all five symptoms, including fever, as binary targets. It creates one fixed multi-label-stratified 80% training pool and 20% held-out test set. The test patients never participate in cross-validation, normalization, class weighting, early stopping, or epoch selection.

Within the 80% training pool, nested subsets from 5% through 100% are evaluated. Thus the final models use 4% through 80% of the complete dataset. Every subset uses cross-validation to estimate validation performance and select the final training duration; the final model is retrained on the entire subset and evaluated on the same fixed test set.


## Training and experiment configuration — optionally edit

The default runs five nested learning curves with seeds 42–46. The 20% test split remains fixed, while each seed creates a different reproducible nested order within the same training pool.


In [ ]:
if LEGACY_REPRODUCTION_MODE:
    print('Workflow B configuration skipped in legacy reproduction mode.')
else:
    import copy
    import json
    import os
    import random
    from datetime import datetime, timezone

    import matplotlib.pyplot as plt
    import pandas as pd
    from iterstrat.ml_stratifiers import (
        MultilabelStratifiedKFold,
        MultilabelStratifiedShuffleSplit,
    )
    from sklearn.metrics import (
        accuracy_score,
        precision_recall_fscore_support,
        roc_auc_score,
    )
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    CURRENT_FRAMEWORK_HEAD = "modern_mlp"  # Or "legacy_two_layer_linear".
    RUN_NAME = f"binary_symptom_learning_curve_5seeds_{CURRENT_FRAMEWORK_HEAD}"
    TRAINING_CONFIG = {
        "random_seed": RANDOM_SEED,
        "test_fraction": 0.20,
        "training_pool_fractions": [
            0.05, 0.10, 0.20, 0.30, 0.40, 0.50,
            0.60, 0.70, 0.80, 0.90, 1.00,
        ],
        "cv_folds": 5,
        "experiment_seeds": [42, 43, 44, 45, 46],
        "head_type": CURRENT_FRAMEWORK_HEAD,  # Workflow B only.
        "hidden_dim": 128,
        "legacy_head_dim": 256,
        "dropout": 0.25,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "max_epochs": 100,
        "early_stopping_patience": 12,
        "binary_threshold": 0.5,
    }

    def seed_everything(seed: int) -> None:
        '''Seed Python, NumPy, and PyTorch for reproducible experiments.'''
        os.environ["PYTHONHASHSEED"] = str(seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    seed_everything(RANDOM_SEED)
    TRAIN_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    RUN_CHECKPOINT_DIR = MODEL_CHECKPOINT_DIR / RUN_NAME
    RUN_RESULTS_DIR = EVALUATION_RESULTS_DIR / RUN_NAME
    RUN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    RUN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    BEST_CHECKPOINT_PATH = RUN_CHECKPOINT_DIR / "best_model.pt"  # 100% training-pool alias.

    run_metadata = {
        **TRAINING_CONFIG,
        "run_name": RUN_NAME,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "embedding_model": EMBEDDING_MODEL_NAME,
        "embedding_npz": str(EMBEDDING_NPZ_PATH),
        "label_names": label_names.tolist(),
        "fever_definition": "0=no fever; 1=original low or high fever",
        "training_device": str(TRAIN_DEVICE),
    }
    with open(RUN_CHECKPOINT_DIR / "training_config.json", "w", encoding="utf-8") as handle:
        json.dump(run_metadata, handle, indent=2)
    with open(RUN_CHECKPOINT_DIR / "random_seed.txt", "w", encoding="utf-8") as handle:
        handle.write(f"{RANDOM_SEED}\n")

    print(f"Training device: {TRAIN_DEVICE}")
    print(f"Run checkpoint directory: {RUN_CHECKPOINT_DIR}")
    print(f"Run results directory: {RUN_RESULTS_DIR}")
    print(f"Learning-curve fractions: {TRAINING_CONFIG['training_pool_fractions']}")


## Create the fixed 80%/20% split and nested training subsets

The split is stratified across all five binary labels. Nested subset orderings are also constructed from stratified 5%-sized chunks, so a larger subset contains every patient from the smaller subset for the same experiment seed.


In [ ]:
if LEGACY_REPRODUCTION_MODE:
    print('Workflow B split construction skipped in legacy reproduction mode.')
else:
    all_indices = np.arange(len(patient_ids))
    outer_splitter = MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=TRAINING_CONFIG["test_fraction"],
        random_state=RANDOM_SEED,
    )
    train_pool_position, test_position = next(outer_splitter.split(X, y))
    train_pool_indices = all_indices[train_pool_position]
    test_indices = all_indices[test_position]

    if len(train_pool_indices) < 20:
        raise ValueError(
            "At least 20 training-pool patients are required for 5% nested increments."
        )

    def make_nested_training_order(indices: np.ndarray, seed: int) -> np.ndarray:
        '''Return a reproducible, approximately multi-label-stratified nested order.'''
        chunk_splitter = MultilabelStratifiedKFold(
            n_splits=20,
            shuffle=True,
            random_state=seed,
        )
        chunks = []
        local_X = X[indices]
        local_y = y[indices]
        for _, chunk_positions in chunk_splitter.split(local_X, local_y):
            chunks.append(indices[chunk_positions])
        order = np.concatenate(chunks)
        if len(order) != len(indices) or len(np.unique(order)) != len(indices):
            raise RuntimeError("Nested training order lost or duplicated patients.")
        return order

    nested_orders = {
        int(seed): make_nested_training_order(train_pool_indices, int(seed))
        for seed in TRAINING_CONFIG["experiment_seeds"]
    }

    split_payload = {
        "train_pool_patient_ids": patient_ids[train_pool_indices],
        "test_patient_ids": patient_ids[test_indices],
    }
    for seed, order in nested_orders.items():
        split_payload[f"nested_training_order_seed_{seed}"] = patient_ids[order]
    np.savez_compressed(RUN_RESULTS_DIR / "data_split_patient_ids.npz", **split_payload)

    print(
        f"Fixed split sizes — training pool: {len(train_pool_indices)} "
        f"({len(train_pool_indices) / len(all_indices):.1%}), "
        f"test: {len(test_indices)} ({len(test_indices) / len(all_indices):.1%})"
    )
    print("Training-pool positive counts:", dict(zip(label_names, y[train_pool_indices].sum(0))))
    print("Test positive counts:", dict(zip(label_names, y[test_indices].sum(0))))


## Define the binary multi-task model, training loop, and metrics

Normalization statistics and positive-class weights are recomputed using only the training patients in each CV fold or final subset. This prevents validation/test leakage. All five outputs use sigmoid probabilities and binary cross-entropy.


In [ ]:
if LEGACY_REPRODUCTION_MODE:
    print('Workflow B model utilities skipped in legacy reproduction mode.')
else:
    class LegacyHeadOnlyModel(nn.Module):
        """Two-layer linear head; no activation, dropout, or normalization."""

        def __init__(self, input_dim: int, num_labels: int = 5, head_dim: int = 256) -> None:
            super().__init__()
            self.projector = nn.Linear(input_dim, head_dim)
            self.classifier = nn.Linear(head_dim, num_labels)

        def forward(self, features: torch.Tensor) -> torch.Tensor:
            return self.classifier(self.projector(features))


    class SymptomMultiTaskMLP(nn.Module):
        '''One 768-to-128 hidden layer and five binary symptom outputs.'''

        def __init__(self, input_dim: int, hidden_dim: int, dropout: float) -> None:
            super().__init__()
            self.shared = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            self.output_head = nn.Linear(hidden_dim, len(label_names))

        def forward(self, features: torch.Tensor) -> torch.Tensor:
            return self.output_head(self.shared(features))


    def feature_scaler(indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        mean = X[indices].mean(axis=0, dtype=np.float64).astype(np.float32)
        std = X[indices].std(axis=0, dtype=np.float64).astype(np.float32)
        std[std < 1e-8] = 1.0
        return mean, std


    def positive_class_weights(indices: np.ndarray) -> torch.Tensor:
        positive = y[indices].sum(axis=0).astype(np.float32)
        negative = len(indices) - positive
        if np.any(positive == 0):
            missing = label_names[positive == 0].tolist()
            print(f"WARNING: no positive training examples for {missing}; using finite weights.")
        weights = negative / np.maximum(positive, 1.0)
        return torch.from_numpy(weights).to(TRAIN_DEVICE)


    def make_loader(
        indices: np.ndarray,
        mean: np.ndarray,
        std: np.ndarray,
        shuffle: bool,
        seed: int,
    ) -> DataLoader:
        scaled_features = ((X[indices] - mean) / std).astype(np.float32)
        dataset = TensorDataset(
            torch.from_numpy(scaled_features),
            torch.from_numpy(y[indices]),
        )
        generator = torch.Generator().manual_seed(seed)
        return DataLoader(
            dataset,
            batch_size=TRAINING_CONFIG["batch_size"],
            shuffle=shuffle,
            generator=generator if shuffle else None,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        )


    def calculate_metrics(
        targets: np.ndarray,
        probabilities: np.ndarray,
        loss: float,
    ) -> tuple[dict, np.ndarray]:
        predictions = (
            probabilities >= TRAINING_CONFIG["binary_threshold"]
        ).astype(np.int64)
        per_label = {}
        f1_scores = []
        for label_index, label_name in enumerate(label_names):
            truth = targets[:, label_index]
            prediction = predictions[:, label_index]
            precision, recall, f1, _ = precision_recall_fscore_support(
                truth, prediction, average="binary", zero_division=0
            )
            try:
                auroc = float(roc_auc_score(truth, probabilities[:, label_index]))
            except ValueError:
                auroc = None
            per_label[str(label_name)] = {
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
                "auroc": auroc,
                "accuracy": float(accuracy_score(truth, prediction)),
            }
            f1_scores.append(float(f1))
        return {
            "loss": float(loss),
            "overall_macro_f1": float(np.mean(f1_scores)),
            "per_label": per_label,
        }, predictions


    def run_epoch(
        model: nn.Module,
        loader: DataLoader,
        loss_function: nn.Module,
        optimizer: torch.optim.Optimizer | None = None,
    ) -> tuple[dict, dict]:
        training = optimizer is not None
        model.train(training)
        total_loss = 0.0
        all_targets, all_probabilities = [], []

        for features, targets in loader:
            features = features.to(TRAIN_DEVICE, non_blocking=True)
            targets = targets.to(TRAIN_DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.set_grad_enabled(training):
                logits = model(features)
                loss = loss_function(logits, targets.float())
                if training:
                    loss.backward()
                    optimizer.step()
            total_loss += loss.item() * len(features)
            all_targets.append(targets.detach().cpu().numpy())
            all_probabilities.append(torch.sigmoid(logits).detach().cpu().numpy())

        targets_array = np.concatenate(all_targets)
        probabilities = np.concatenate(all_probabilities)
        mean_loss = total_loss / len(loader.dataset)
        metrics, predictions = calculate_metrics(targets_array, probabilities, mean_loss)
        return metrics, {
            "targets": targets_array,
            "predictions": predictions,
            "probabilities": probabilities,
        }


    def new_model_and_optimizer(seed: int) -> tuple[nn.Module, torch.optim.Optimizer]:
        seed_everything(seed)
        if TRAINING_CONFIG["head_type"] == "legacy_two_layer_linear":
            # Workflow B architecture comparison only: all modern splitting, scaling,
            # class weighting, CV, and retraining behavior remains active.
            model = LegacyHeadOnlyModel(
                input_dim=X.shape[1],
                num_labels=len(label_names),
                head_dim=TRAINING_CONFIG["legacy_head_dim"],
            ).to(TRAIN_DEVICE)
        elif TRAINING_CONFIG["head_type"] == "modern_mlp":
            model = SymptomMultiTaskMLP(
                input_dim=X.shape[1],
                hidden_dim=TRAINING_CONFIG["hidden_dim"],
                dropout=TRAINING_CONFIG["dropout"],
            ).to(TRAIN_DEVICE)
        else:
            raise ValueError(f"Unknown head_type: {TRAINING_CONFIG['head_type']}")
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=TRAINING_CONFIG["learning_rate"],
            weight_decay=TRAINING_CONFIG["weight_decay"],
        )
        return model, optimizer


## Run cross-validation, retrain each final model, and test

For each subset, the best epoch is found independently in every CV fold using validation macro-F1. The median of those fold-specific best epochs becomes the final training duration. The final model then trains from scratch on the entire subset—without a separate validation holdout—and predicts the untouched fixed 20% test set.

This is the longest cell. With the default single experiment seed it trains 55 CV models plus 11 final models. Add experiment seeds only when the initial run is working and the additional runtime is acceptable.


In [ ]:
if LEGACY_REPRODUCTION_MODE:
    print('Workflow B learning-curve training skipped in legacy reproduction mode.')
else:
    def cross_validate_subset(
        subset_indices: np.ndarray,
        experiment_seed: int,
        training_fraction: float,
    ) -> tuple[list[dict], int]:
        n_splits = min(TRAINING_CONFIG["cv_folds"], len(subset_indices))
        if n_splits < 2:
            raise ValueError("A training subset must contain at least two patients for CV.")
        splitter = MultilabelStratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=experiment_seed,
        )
        fold_rows = []
        best_epochs = []

        for fold, (train_positions, validation_positions) in enumerate(
            splitter.split(X[subset_indices], y[subset_indices]), start=1
        ):
            fold_train_indices = subset_indices[train_positions]
            fold_validation_indices = subset_indices[validation_positions]
            fold_seed = experiment_seed * 1000 + int(round(training_fraction * 100)) * 10 + fold
            mean, std = feature_scaler(fold_train_indices)
            train_loader = make_loader(
                fold_train_indices, mean, std, shuffle=True, seed=fold_seed
            )
            validation_loader = make_loader(
                fold_validation_indices, mean, std, shuffle=False, seed=fold_seed
            )
            loss_function = nn.BCEWithLogitsLoss(
                pos_weight=positive_class_weights(fold_train_indices)
            )
            model, optimizer = new_model_and_optimizer(fold_seed)
            best_score = -np.inf
            best_epoch = 1
            epochs_without_improvement = 0

            for epoch in range(1, TRAINING_CONFIG["max_epochs"] + 1):
                run_epoch(model, train_loader, loss_function, optimizer)
                validation_metrics, _ = run_epoch(model, validation_loader, loss_function)
                score = validation_metrics["overall_macro_f1"]
                if score > best_score:
                    best_score = score
                    best_epoch = epoch
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1
                    if epochs_without_improvement >= TRAINING_CONFIG["early_stopping_patience"]:
                        break

            best_epochs.append(best_epoch)
            fold_rows.append({
                "experiment_seed": experiment_seed,
                "training_pool_fraction": training_fraction,
                "full_dataset_fraction": training_fraction * (1.0 - TRAINING_CONFIG["test_fraction"]),
                "subset_patients": len(subset_indices),
                "fold": fold,
                "fold_train_patients": len(fold_train_indices),
                "fold_validation_patients": len(fold_validation_indices),
                "best_epoch": best_epoch,
                "validation_macro_f1": float(best_score),
            })
            print(
                f"  fold {fold}/{n_splits}: best epoch={best_epoch}, "
                f"validation macro-F1={best_score:.4f}"
            )

        final_epochs = max(1, int(np.median(best_epochs)))
        return fold_rows, final_epochs


    def train_final_subset(
        subset_indices: np.ndarray,
        final_epochs: int,
        seed: int,
    ) -> tuple[nn.Module, np.ndarray, np.ndarray, dict]:
        mean, std = feature_scaler(subset_indices)
        train_loader = make_loader(subset_indices, mean, std, shuffle=True, seed=seed)
        loss_function = nn.BCEWithLogitsLoss(
            pos_weight=positive_class_weights(subset_indices)
        )
        model, optimizer = new_model_and_optimizer(seed)
        train_metrics = {}
        for _ in range(final_epochs):
            train_metrics, _ = run_epoch(model, train_loader, loss_function, optimizer)
        return model, mean, std, train_metrics


    cv_rows = []
    run_rows = []
    per_label_rows = []
    all_metrics = []

    for experiment_seed in TRAINING_CONFIG["experiment_seeds"]:
        experiment_seed = int(experiment_seed)
        nested_order = nested_orders[experiment_seed]
        for training_fraction in TRAINING_CONFIG["training_pool_fractions"]:
            training_fraction = float(training_fraction)
            subset_size = max(
                2,
                min(len(nested_order), int(round(len(nested_order) * training_fraction))),
            )
            subset_indices = nested_order[:subset_size]
            percent = int(round(training_fraction * 100))
            print(
                f"\nSeed {experiment_seed} | {percent}% of training pool | "
                f"{subset_size} patients ({subset_size / len(all_indices):.1%} of full data)"
            )

            fold_rows, final_epochs = cross_validate_subset(
                subset_indices, experiment_seed, training_fraction
            )
            cv_rows.extend(fold_rows)
            fold_scores = np.array(
                [row["validation_macro_f1"] for row in fold_rows], dtype=float
            )

            final_seed = experiment_seed * 1000 + percent
            model, mean, std, train_metrics = train_final_subset(
                subset_indices, final_epochs, final_seed
            )
            test_loader = make_loader(
                test_indices, mean, std, shuffle=False, seed=final_seed
            )
            all_dataset_loader = make_loader(
                all_indices, mean, std, shuffle=False, seed=final_seed
            )
            test_loss_function = nn.BCEWithLogitsLoss(
                pos_weight=positive_class_weights(subset_indices)
            )
            test_metrics, test_arrays = run_epoch(model, test_loader, test_loss_function)
            all_dataset_metrics, all_dataset_arrays = run_epoch(
                model, all_dataset_loader, test_loss_function
            )

            checkpoint_path = RUN_CHECKPOINT_DIR / (
                f"model_trainpool_{percent:03d}pct_seed{experiment_seed}.pt"
            )
            checkpoint_payload = {
                "model_state_dict": copy.deepcopy(model.state_dict()),
                "training_pool_fraction": training_fraction,
                "full_dataset_fraction": subset_size / len(all_indices),
                "experiment_seed": experiment_seed,
                "final_epochs": final_epochs,
                "cv_validation_macro_f1_mean": float(fold_scores.mean()),
                "cv_validation_macro_f1_std": float(fold_scores.std(ddof=1)) if len(fold_scores) > 1 else 0.0,
                "all_dataset_metrics": all_dataset_metrics,
                "test_metrics": test_metrics,
                "training_config": TRAINING_CONFIG,
                "label_names": label_names.tolist(),
                "feature_mean": mean,
                "feature_std": std,
                "fever_definition": "0=no fever; 1=original low or high fever",
            }
            torch.save(checkpoint_payload, checkpoint_path)

            prediction_path = RUN_RESULTS_DIR / (
                f"test_predictions_trainpool_{percent:03d}pct_seed{experiment_seed}.npz"
            )
            np.savez_compressed(
                prediction_path,
                patient_ids=patient_ids[test_indices],
                y_true=test_arrays["targets"].astype(np.int64),
                y_pred=test_arrays["predictions"].astype(np.int64),
                probabilities=test_arrays["probabilities"].astype(np.float32),
                label_names=label_names,
                training_pool_fraction=np.float64(training_fraction),
                full_dataset_fraction=np.float64(subset_size / len(all_indices)),
                experiment_seed=np.int64(experiment_seed),
            )
            all_dataset_prediction_path = RUN_RESULTS_DIR / (
                f"all_dataset_predictions_trainpool_{percent:03d}pct_seed{experiment_seed}.npz"
            )
            np.savez_compressed(
                all_dataset_prediction_path,
                patient_ids=patient_ids[all_indices],
                y_true=all_dataset_arrays["targets"].astype(np.int64),
                y_pred=all_dataset_arrays["predictions"].astype(np.int64),
                probabilities=all_dataset_arrays["probabilities"].astype(np.float32),
                label_names=label_names,
                training_pool_fraction=np.float64(training_fraction),
                full_dataset_fraction=np.float64(subset_size / len(all_indices)),
                experiment_seed=np.int64(experiment_seed),
            )

            # Convenient backward-compatible aliases point to the first 100% model.
            if training_fraction == 1.0 and experiment_seed == int(TRAINING_CONFIG["experiment_seeds"][0]):
                torch.save(checkpoint_payload, BEST_CHECKPOINT_PATH)
                np.savez_compressed(
                    RUN_RESULTS_DIR / "test_predictions.npz",
                    patient_ids=patient_ids[test_indices],
                    y_true=test_arrays["targets"].astype(np.int64),
                    y_pred=test_arrays["predictions"].astype(np.int64),
                    probabilities=test_arrays["probabilities"].astype(np.float32),
                    label_names=label_names,
                )
                np.savez_compressed(
                    RUN_RESULTS_DIR / "all_dataset_predictions.npz",
                    patient_ids=patient_ids[all_indices],
                    y_true=all_dataset_arrays["targets"].astype(np.int64),
                    y_pred=all_dataset_arrays["predictions"].astype(np.int64),
                    probabilities=all_dataset_arrays["probabilities"].astype(np.float32),
                    label_names=label_names,
                )

            run_row = {
                "experiment_seed": experiment_seed,
                "training_pool_fraction": training_fraction,
                "full_dataset_fraction": subset_size / len(all_indices),
                "subset_patients": subset_size,
                "cv_validation_macro_f1_mean": float(fold_scores.mean()),
                "cv_validation_macro_f1_std": float(fold_scores.std(ddof=1)) if len(fold_scores) > 1 else 0.0,
                "final_epochs": final_epochs,
                "final_train_macro_f1": train_metrics["overall_macro_f1"],
                "all_dataset_macro_f1": all_dataset_metrics["overall_macro_f1"],
                "test_macro_f1": test_metrics["overall_macro_f1"],
                "checkpoint": str(checkpoint_path),
                "test_predictions": str(prediction_path),
                "all_dataset_predictions": str(all_dataset_prediction_path),
            }
            run_rows.append(run_row)
            for evaluation_scope, scope_metrics in (
                ("all_dataset", all_dataset_metrics),
                ("test", test_metrics),
            ):
                for label_name, metrics in scope_metrics["per_label"].items():
                    per_label_rows.append({
                        "evaluation_scope": evaluation_scope,
                        "experiment_seed": experiment_seed,
                        "training_pool_fraction": training_fraction,
                        "full_dataset_fraction": subset_size / len(all_indices),
                        "label": label_name,
                        "macro_f1": scope_metrics["overall_macro_f1"],
                        **metrics,
                    })
            all_metrics.append({
                **run_row,
                "all_dataset": all_dataset_metrics,
                "test": test_metrics,
            })
            print(
                f"  CV mean±SD={fold_scores.mean():.4f}±{run_row['cv_validation_macro_f1_std']:.4f} | "
                f"final epochs={final_epochs} | "
                f"all-dataset macro-F1={all_dataset_metrics['overall_macro_f1']:.4f} | "
                f"test macro-F1={test_metrics['overall_macro_f1']:.4f}"
            )


## Save tables, consolidated metrics, and the learning-curve plot

The plot's x-axis shows the fraction of the complete dataset used by each final model: 4% through 80%. CV and test results are shown separately.


In [ ]:
if LEGACY_REPRODUCTION_MODE:
    print('Workflow B result aggregation skipped in legacy reproduction mode.')
else:
    cv_frame = pd.DataFrame(cv_rows)
    run_frame = pd.DataFrame(run_rows)
    per_label_frame = pd.DataFrame(per_label_rows)
    all_dataset_per_label_frame = per_label_frame.loc[
        per_label_frame["evaluation_scope"] == "all_dataset"
    ].reset_index(drop=True)
    test_per_label_frame = per_label_frame.loc[
        per_label_frame["evaluation_scope"] == "test"
    ].reset_index(drop=True)

    cv_frame.to_csv(RUN_RESULTS_DIR / "cross_validation_folds.csv", index=False)
    run_frame.to_csv(RUN_RESULTS_DIR / "learning_curve_runs.csv", index=False)
    per_label_frame.to_csv(RUN_RESULTS_DIR / "learning_curve_per_label.csv", index=False)
    all_dataset_per_label_frame.to_csv(
        RUN_RESULTS_DIR / "all_dataset_per_label_metrics.csv", index=False
    )
    test_per_label_frame.to_csv(
        RUN_RESULTS_DIR / "test_per_label_metrics.csv", index=False
    )

    summary_frame = (
        run_frame.groupby(
            ["training_pool_fraction", "full_dataset_fraction", "subset_patients"],
            as_index=False,
        )
        .agg(
            cv_validation_macro_f1_mean=("cv_validation_macro_f1_mean", "mean"),
            cv_validation_macro_f1_between_seed_std=("cv_validation_macro_f1_mean", "std"),
            all_dataset_macro_f1_mean=("all_dataset_macro_f1", "mean"),
            all_dataset_macro_f1_std=("all_dataset_macro_f1", "std"),
            test_macro_f1_mean=("test_macro_f1", "mean"),
            test_macro_f1_std=("test_macro_f1", "std"),
            final_epochs_median=("final_epochs", "median"),
            experiment_runs=("experiment_seed", "count"),
        )
        .fillna(0.0)
    )
    summary_frame.to_csv(RUN_RESULTS_DIR / "learning_curve_summary.csv", index=False)

    symptom_f1_summary_frame = (
        per_label_frame.groupby(
            [
                "evaluation_scope",
                "training_pool_fraction",
                "full_dataset_fraction",
                "label",
            ],
            as_index=False,
        )
        .agg(
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            experiment_runs=("experiment_seed", "count"),
        )
        .fillna(0.0)
    )
    symptom_f1_summary_frame.to_csv(
        RUN_RESULTS_DIR / "symptom_f1_summary.csv", index=False
    )

    metrics_payload = {
        "run_name": RUN_NAME,
        "random_seed": RANDOM_SEED,
        "fixed_train_pool_patients": int(len(train_pool_indices)),
        "fixed_test_patients": int(len(test_indices)),
        "fever_definition": "0=no fever; 1=original low or high fever",
        "all_dataset_metric_note": (
            "Descriptive only: the all-dataset evaluation includes patients used for training. "
            "Use fixed-test metrics to assess generalization."
        ),
        "runs": all_metrics,
    }
    with open(RUN_RESULTS_DIR / "evaluation_metrics.json", "w", encoding="utf-8") as handle:
        json.dump(metrics_payload, handle, indent=2)

    test_symptom_f1_frame = symptom_f1_summary_frame.loc[
        symptom_f1_summary_frame["evaluation_scope"] == "test"
    ].sort_values(["label", "full_dataset_fraction"])

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for label_name, label_frame in test_symptom_f1_frame.groupby("label", sort=False):
        x_percent = label_frame["full_dataset_fraction"].to_numpy() * 100
        f1_mean = label_frame["f1_mean"].to_numpy()
        f1_std = label_frame["f1_std"].to_numpy()
        line = ax.plot(x_percent, f1_mean, marker="o", label=label_name)[0]
        if len(TRAINING_CONFIG["experiment_seeds"]) > 1:
            ax.fill_between(
                x_percent,
                np.clip(f1_mean - f1_std, 0, 1),
                np.clip(f1_mean + f1_std, 0, 1),
                color=line.get_color(),
                alpha=0.12,
            )
    ax.set_xlabel("Final training data (% of complete dataset)")
    ax.set_ylabel("F1 score")
    ax.set_title("Per-symptom learning curves (fixed 20% test set)")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plot_path = RUN_RESULTS_DIR / "symptom_f1_learning_curve.png"
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()

    display(summary_frame)
    display(test_symptom_f1_frame)
    print(f"Summary: {RUN_RESULTS_DIR / 'learning_curve_summary.csv'}")
    print(f"Per-symptom F1 summary: {RUN_RESULTS_DIR / 'symptom_f1_summary.csv'}")
    print(f"All-dataset per-label metrics: {RUN_RESULTS_DIR / 'all_dataset_per_label_metrics.csv'}")
    print(f"Test per-label metrics: {RUN_RESULTS_DIR / 'test_per_label_metrics.csv'}")
    print(f"All metrics: {RUN_RESULTS_DIR / 'evaluation_metrics.json'}")
    print(f"Plot: {plot_path}")
    print(f"100% training-pool checkpoint alias: {BEST_CHECKPOINT_PATH}")
    print(f"100% training-pool test predictions alias: {RUN_RESULTS_DIR / 'test_predictions.npz'}")
    print(f"100% training-pool all-dataset predictions alias: {RUN_RESULTS_DIR / 'all_dataset_predictions.npz'}")


## Saved artifacts

Workflow A writes only under `legacy_reproduction/`: separate legacy-compatible `.npy` embedding/label/ID caches, the best-validation-loss checkpoint and history, split patient IDs, test/all per-patient predictions, per-symptom metrics, metadata JSON, and a run report.

Workflow B retains the existing modern embedding, checkpoint, prediction, metric-table, JSON, and plot outputs. Selecting the legacy head inside Workflow B changes only its architecture and must not be described as reproduction.